In [0]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.ml.functions import vector_to_array

# Récupère la session Spark active
spark = SparkSession.builder.getOrCreate()

# Récupère dbutils de façon compatible (Databricks SDK / PySpark)
try:
    from databricks.sdk.runtime import dbutils
except ImportError:
    try:
        from pyspark.dbutils import DBUtils
        dbutils = DBUtils(spark)
    except ImportError:
        pass  # En environnement Databricks natif, dbutils est déjà injecté

# Contexte UC
spark.sql("USE CATALOG main")
spark.sql("USE SCHEMA gold")

# Charger les prédictions streaming
pred_df = spark.read.table("main.gold.fraud_predictions_stream")

# Accuracy globale
accuracy_df = (
    pred_df
    .select(
        F.sum(F.when(F.col("prediction") == F.col("is_fraud"), 1).otherwise(0)).alias("correct"),  # noqa: E501
        F.count("*").alias("total")
    )
    .withColumn("accuracy", F.col("correct") / F.col("total"))
)

# Faux positifs / faux négatifs
rates_df = (
    pred_df
    .select(
        F.sum(F.when((F.col("prediction") == 1) & (F.col("is_fraud") == 0), 1).otherwise(0)).alias("fp"),  # noqa: E501
        F.sum(F.when((F.col("prediction") == 0) & (F.col("is_fraud") == 1), 1).otherwise(0)).alias("fn"),  # noqa: E501
        F.count("*").alias("total")
    )
    .withColumn("false_positive_rate", F.col("fp") / F.col("total"))
    .withColumn("false_negative_rate", F.col("fn") / F.col("total"))
)

# Probabilité moyenne de fraude
prob_df = (
    pred_df
    .select(F.avg(vector_to_array(F.col("probability"))[1]).alias("avg_fraud_probability"))  # noqa: E501
)

# Transactions mal classées (erreurs du modèle)
errors_df = (
    pred_df
    .filter(F.col("prediction") != F.col("is_fraud"))
    .orderBy(F.col("time").desc())
)

# Qualité par heure (dérive temporelle)
hour_quality_df = (
    pred_df
    .withColumn("hour", (F.col("time") / 3600).cast("int"))
    .groupBy("hour")
    .agg(
        F.avg(F.when(F.col("prediction") == F.col("is_fraud"), 1).otherwise(0)).alias("hour_accuracy"),  # noqa: E501
        F.avg(vector_to_array(F.col("probability"))[1]).alias("avg_fraud_prob")
    )
    .orderBy("hour")
)

display(hour_quality_df)

print("Monitoring du modèle terminé : accuracy, FP/FN, probas, dérive par heure.")